# LeetCode 1824 — Minimum Sideway Jumps bằng Python

Link đề: https://leetcode.com/problems/minimum-sideway-jumps/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minSideJumps(self, obstacles: List[int]) -> int:
        ...
```

Lộ trình: **vét cạn mọi cách đi** (DFS, chỉ cho `n` nhỏ) → **đồ thị trạng thái `(điểm, làn)` + 0-1 BFS** `O(n)` → **DP ba làn cuộn** `O(n)` thời gian, `O(1)` bộ nhớ (lời giải nộp) → **tham lam "nhảy sang làn có chướng ngại xa nhất"**.

## 1. Mục tiêu học

1. Mô hình hóa bài toán di chuyển trên lưới `3 × (n+1)` thành **đồ thị trạng thái** `(điểm, làn)` với hai loại cạnh: **đi thẳng** (chi phí 0) và **nhảy ngang** (chi phí 1).
2. Dùng **0-1 BFS** (deque: cạnh 0 đẩy đầu, cạnh 1 đẩy cuối) cho đồ thị trọng số 0/1.
3. Viết **DP cuộn ba trạng thái** `dp[làn]`, nắm đúng **thứ tự cập nhật**: chặn làn có chướng ngại **trước**, rồi mới xét nhảy ngang.
4. Hiểu và chứng minh tham lam "nhảy sang làn mà chướng ngại kế tiếp ở **xa nhất**".
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed; tránh lỗi "nhảy chỉ sang làn kề", "khởi tạo sai", "đệ quy sâu `5·10⁵`".

## 2. Đề bài & ví dụ

Có một con đường **3 làn** dài `n`, gồm `n + 1` điểm đánh số `0..n`. Một con ếch bắt đầu ở **điểm 0, làn 2** và muốn tới **điểm `n`** (làn nào cũng được). Trên đường có thể có chướng ngại vật.

Cho mảng `obstacles` độ dài `n + 1`: `obstacles[i]` (từ 0 đến 3) là **làn có chướng ngại** tại điểm `i`; `obstacles[i] == 0` nghĩa là điểm `i` không có chướng ngại. Mỗi điểm có **tối đa một** chướng ngại trên 3 làn.

- Ếch đi từ điểm `i` tới `i + 1` **cùng làn** nếu làn đó **không** có chướng ngại tại điểm `i + 1`.
- Để tránh chướng ngại, ếch có thể **nhảy ngang** sang làn khác (**không cần kề nhau**) **tại cùng một điểm**, nếu làn mới không có chướng ngại tại điểm đó. Ví dụ từ làn 3 tại điểm 3 nhảy sang làn 1 tại điểm 3.

Trả về **số lần nhảy ngang ít nhất** để tới điểm `n` (làn bất kỳ), xuất phát từ làn 2 tại điểm 0. Đề bảo đảm điểm `0` và điểm `n` không có chướng ngại.

**Ví dụ 1:** `obstacles = [0,1,2,3,0]` → `2`.

**Ví dụ 2:** `obstacles = [0,1,1,3,3,0]` → `0` (làn 2 không có chướng ngại).

**Ví dụ 3:** `obstacles = [0,2,1,0,3,0]` → `2`.

**Ràng buộc:** `obstacles.length == n + 1`, `1 ≤ n ≤ 5·10⁵`, `0 ≤ obstacles[i] ≤ 3`, `obstacles[0] == obstacles[n] == 0`.

In [1]:
import random
import sys
import time
from collections import deque
from functools import lru_cache
from math import inf
from typing import List

examples = [([0, 1, 2, 3, 0], 2), ([0, 1, 1, 3, 3, 0], 0), ([0, 2, 1, 0, 3, 0], 2)]

def draw(obstacles):
    """Vẽ đường: hàng = làn 1..3, cột = điểm; '#' là chướng ngại."""
    for lane in (1, 2, 3):
        print(f"làn {lane}: " + " ".join("#" if v == lane else "." for v in obstacles))

for obs, want in examples:
    print(obs, "-> kỳ vọng", want); draw(obs); print()

[0, 1, 2, 3, 0] -> kỳ vọng 2
làn 1: . # . . .
làn 2: . . # . .
làn 3: . . . # .

[0, 1, 1, 3, 3, 0] -> kỳ vọng 0
làn 1: . # # . . .
làn 2: . . . . . .
làn 3: . . . # # .

[0, 2, 1, 0, 3, 0] -> kỳ vọng 2
làn 1: . . # . . .
làn 2: . # . . . .
làn 3: . . . . # .



## 3. Kiến thức nền

### 3.1 Trạng thái `(điểm, làn)` và hai loại bước

Ở trạng thái `(i, l)`:
- **đi thẳng** tới `(i+1, l)` nếu `obstacles[i+1] != l` — chi phí 0;
- **nhảy ngang** tới `(i, l')` với `l' ≠ l` nếu `obstacles[i] != l'` — chi phí 1.

In [2]:
def moves(obstacles, i, l):
    out = []
    if i + 1 < len(obstacles) and obstacles[i + 1] != l:
        out.append(((i + 1, l), 0))
    for l2 in (1, 2, 3):
        if l2 != l and obstacles[i] != l2:
            out.append(((i, l2), 1))
    return out

print("từ (0, làn 2) trên Ví dụ 1:", moves([0, 1, 2, 3, 0], 0, 2))
print("từ (1, làn 3) trên Ví dụ 1:", moves([0, 1, 2, 3, 0], 1, 3))

từ (0, làn 2) trên Ví dụ 1: [((1, 2), 0), ((0, 1), 1), ((0, 3), 1)]
từ (1, làn 3) trên Ví dụ 1: [((2, 3), 0), ((1, 2), 1)]


### 3.2 `deque` cho 0-1 BFS

Với đồ thị chỉ có trọng số 0 và 1: cạnh 0 → `appendleft` (xử lý ngay), cạnh 1 → `append` (xử lý sau). Hàng đợi luôn chứa các khoảng cách `d` và `d + 1` theo đúng thứ tự — như Dijkstra nhưng `O(V + E)`.

In [3]:
dq = deque()
dq.append(("B", 1)); dq.appendleft(("A", 0)); dq.append(("C", 1))
print(list(dq))

[('A', 0), ('B', 1), ('C', 1)]


### 3.3 `math.inf` làm "không tới được"

`inf + 1 == inf` và `min(inf, x) == x` — tiện để đánh dấu làn bị chặn trong DP.

In [4]:
print(inf + 1 == inf, min(inf, 3), min([inf, 2, inf]) + 1)

True 3 3


### 3.4 Chỉ số làn 1..3 trong list 0..2

Ta lưu `dp[0], dp[1], dp[2]` cho làn 1, 2, 3. Chướng ngại `v` (1..3) chặn `dp[v − 1]`; `v = 0` không chặn gì.

In [5]:
dp = [1, 0, 1]          # tại điểm 0: làn 2 tốn 0, làn 1 và 3 tốn 1 (nhảy ngang ngay tại điểm 0)
v = 2
if v:
    dp[v - 1] = inf
print(dp)

[1, inf, 1]


### 3.5 Quét ngược để biết "chướng ngại kế tiếp" của mỗi làn

`nxt[l]` = điểm gần nhất `≥ i` có chướng ngại trên làn `l`. Quét từ phải sang trái cập nhật được trong `O(n)` — dùng cho cách tham lam.

In [6]:
obs = [0, 2, 1, 0, 3, 0]
n = len(obs) - 1
nxt = {1: n + 1, 2: n + 1, 3: n + 1}
table = []
for i in range(n, -1, -1):
    if obs[i]:
        nxt[obs[i]] = i
    table.append((i, dict(nxt)))
for i, d in reversed(table):
    print(f"điểm {i}: chướng ngại kế tiếp {d}")

điểm 0: chướng ngại kế tiếp {1: 2, 2: 1, 3: 4}
điểm 1: chướng ngại kế tiếp {1: 2, 2: 1, 3: 4}
điểm 2: chướng ngại kế tiếp {1: 2, 2: 6, 3: 4}
điểm 3: chướng ngại kế tiếp {1: 6, 2: 6, 3: 4}
điểm 4: chướng ngại kế tiếp {1: 6, 2: 6, 3: 4}
điểm 5: chướng ngại kế tiếp {1: 6, 2: 6, 3: 6}


## 4. Ý tưởng nền tảng / trực giác

Vẽ Ví dụ 3 `obstacles = [0,2,1,0,3,0]`:

```
điểm:   0 1 2 3 4 5
làn 1:  . . # . . .
làn 2:  E # . . . .      E = ếch
làn 3:  . . . . # .
```

Ếch ở làn 2, điểm 1 làn 2 bị chặn ⇒ phải nhảy ngang ở điểm 0. Nhảy sang làn 1 thì gặp chặn ở điểm 2; nhảy sang làn 3 thì gặp chặn ở điểm 4 (xa hơn). Chọn làn 3: đi tới điểm 3, trước điểm 4 phải nhảy (sang làn 1 hoặc 2, cả hai trống tới cuối) ⇒ tổng **2** lần.

Nếu chọn làn 1 ở điểm 0: đi tới điểm 1, phải nhảy ở điểm 1 (sang làn 3, vì làn 2 bị chặn tại điểm 1) rồi lại phải nhảy trước điểm 4 ⇒ 3 lần. Vậy **chọn làn để nhảy** là quyết định quan trọng.

Tư duy DP: tại mỗi điểm, nhớ "số lần nhảy ít nhất để đứng ở làn 1 / 2 / 3 tại điểm này".

In [7]:
draw([0, 2, 1, 0, 3, 0])

làn 1: . . # . . .
làn 2: . # . . . .
làn 3: . . . . # .


## 5. Cách 1 — Brute Force / baseline: DFS thử mọi cách đi

**Ý tưởng:** đệ quy trên `(i, l, đã_nhảy_ở_điểm_này)`: hoặc đi thẳng (nếu được), hoặc nhảy ngang sang một làn khác (chỉ cho nhảy **một lần** mỗi điểm — nhảy hai lần liên tiếp tại cùng điểm không bao giờ có lợi vì nhảy thẳng sang làn đích chỉ tốn 1). Không ghi nhớ ⇒ thử **mọi** đường, chắc chắn đúng nhưng hàm mũ.

In [8]:
def side_jumps_brute(obstacles):
    n = len(obstacles) - 1
    best = [inf]
    def dfs(i, l, cost, jumped):
        if cost >= best[0]:
            return
        if i == n:
            best[0] = cost; return
        if obstacles[i + 1] != l:
            dfs(i + 1, l, cost, False)
        if not jumped:
            for l2 in (1, 2, 3):
                if l2 != l and obstacles[i] != l2:
                    dfs(i, l2, cost + 1, True)
    dfs(0, 2, 0, False)
    return best[0]

for obs, want in examples:
    print(side_jumps_brute(obs), "| kỳ vọng", want)

2 | kỳ vọng 2
0 | kỳ vọng 0
2 | kỳ vọng 2


**Độ phức tạp:** mỗi điểm có tới 3 lựa chọn (đi thẳng hoặc nhảy sang 2 làn) ⇒ cỡ `O(3ⁿ)` (cắt nhánh `cost ≥ best` giúp nhiều nhưng không đổi bậc). Đệ quy sâu `n`. Chỉ dùng cho `n ≤ 15`.

**Nút thắt:** cùng một trạng thái `(i, l)` được thăm lại theo nhiều đường; chỉ cần nhớ **chi phí nhỏ nhất** tới mỗi trạng thái ⇒ BFS/DP.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đồ thị):** có `3(n+1)` trạng thái, mỗi trạng thái ≤ 3 cạnh, trọng số 0/1 ⇒ **0-1 BFS** cho đáp án trong `O(n)`.

**Insight 2 (DP theo điểm):** vì ếch chỉ đi **tiến**, xử lý điểm theo thứ tự `0, 1, …, n`. Gọi `dp[l]` = số lần nhảy ít nhất để đứng ở làn `l` **tại điểm hiện tại**. Chuyển từ điểm `i − 1` sang `i`:

1. *Đi thẳng:* `dp[l]` giữ nguyên (đi từ `(i−1, l)` tới `(i, l)`), **trừ** làn `obstacles[i]` không đứng được ⇒ đặt `dp[obstacles[i]] = inf`.
2. *Nhảy ngang tại điểm `i`:* với mỗi làn `l` không bị chặn tại `i`: `dp[l] = min(dp[l], min(dp) + 1)`.

Thứ tự **bắt buộc**: bước 1 trước bước 2 — nếu không, `min(dp)` có thể lấy giá trị của làn đang bị chặn.

Tại sao "đi thẳng" chỉ cần điều kiện tại điểm `i` (không cần xét điểm `i − 1`)? Vì `dp[l]` hữu hạn ở điểm `i − 1` đã nghĩa là đứng được ở `(i − 1, l)`; đi tới `(i, l)` chỉ cần ô đích trống.

**Khởi tạo:** tại điểm 0, `dp = [1, 0, 1]` (làn 2 miễn phí; làn 1, 3 tốn một lần nhảy ngang tại điểm 0, luôn hợp lệ vì điểm 0 không có chướng ngại).

**Insight 3 (tham lam):** chỉ nhảy khi **bắt buộc** (ô phía trước bị chặn), và khi nhảy thì chọn làn có **chướng ngại kế tiếp xa nhất**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — 0-1 BFS trên đồ thị trạng thái (`O(n)`)

In [9]:
def side_jumps_01bfs(obstacles):
    n = len(obstacles) - 1
    dist = [[inf] * 4 for _ in range(n + 1)]       # dist[i][l], l = 1..3
    dist[0][2] = 0
    dq = deque([(0, 0, 2)])
    while dq:
        d, i, l = dq.popleft()
        if d > dist[i][l]:
            continue
        if i == n:
            return d                                 # lần đầu lấy ra điểm n là tối ưu
        if obstacles[i + 1] != l and d < dist[i + 1][l]:
            dist[i + 1][l] = d; dq.appendleft((d, i + 1, l))
        for l2 in (1, 2, 3):
            if l2 != l and obstacles[i] != l2 and d + 1 < dist[i][l2]:
                dist[i][l2] = d + 1; dq.append((d + 1, i, l2))
    return -1

for obs, want in examples:
    print(side_jumps_01bfs(obs), want)

2 2
0 0
2 2


**Độ phức tạp:** `O(V + E) = O(n)` thời gian, `O(n)` bộ nhớ cho `dist`. Tổng quát, dễ mở rộng (thêm luật mới chỉ cần thêm cạnh), nhưng tốn bộ nhớ hơn DP.

### 7.2 Cách 3 — DP ba làn cuộn (`O(n)` thời gian, `O(1)` bộ nhớ) — lời giải chuẩn

In [10]:
def side_jumps_dp_demo(obstacles):
    dp = [1, 0, 1]
    print(f"điểm 0: dp = {dp}")
    for i in range(1, len(obstacles)):
        v = obstacles[i]
        if v:
            dp[v - 1] = inf                          # bước 1: làn bị chặn tại điểm i
        best = min(dp) + 1
        for l in range(3):
            if l != v - 1:
                dp[l] = min(dp[l], best)             # bước 2: nhảy ngang tại điểm i
        print(f"điểm {i} (chặn làn {v or '-'}): dp = {[x if x != inf else '∞' for x in dp]}")
    return min(dp)

print("=>", side_jumps_dp_demo([0, 2, 1, 0, 3, 0]))

điểm 0: dp = [1, 0, 1]
điểm 1 (chặn làn 2): dp = [1, '∞', 1]
điểm 2 (chặn làn 1): dp = ['∞', 2, 1]
điểm 3 (chặn làn -): dp = [2, 2, 1]
điểm 4 (chặn làn 3): dp = [2, 2, '∞']
điểm 5 (chặn làn -): dp = [2, 2, 3]
=> 2


In [11]:
class SolutionDP:
    def minSideJumps(self, obstacles: List[int]) -> int:
        dp = [1, 0, 1]
        for v in obstacles[1:]:
            if v:
                dp[v - 1] = inf
            best = min(dp) + 1
            for l in range(3):
                if l != v - 1 and dp[l] > best:
                    dp[l] = best
        return min(dp)

for obs, want in examples:
    print(SolutionDP().minSideJumps(obs), want)

2 2
0 0
2 2


**Độ phức tạp:** mỗi điểm `O(3)` ⇒ `O(n)` thời gian, `O(1)` bộ nhớ. (Chú ý `l != v − 1`: khi `v = 0` thì `v − 1 = −1` không trùng làn nào — đúng ý "không làn nào bị chặn".)

### 7.3 Cách 4 — Tham lam: nhảy khi bắt buộc, chọn làn có chướng ngại xa nhất

Cần biết, tại điểm `i`, chướng ngại kế tiếp của mỗi làn ở đâu. Quét ngược một lần để có `nxt[i][l]`; hoặc tiết kiệm bộ nhớ: lưu danh sách vị trí chướng ngại theo làn và dùng con trỏ tiến.

In [12]:
class SolutionGreedy:
    def minSideJumps(self, obstacles: List[int]) -> int:
        n = len(obstacles) - 1
        pos = {1: [], 2: [], 3: []}
        for i, v in enumerate(obstacles):
            if v:
                pos[v].append(i)
        ptr = {1: 0, 2: 0, 3: 0}
        def next_obstacle(l, i):                     # điểm chướng ngại đầu tiên > i trên làn l
            p = pos[l]
            while ptr[l] < len(p) and p[ptr[l]] <= i:
                ptr[l] += 1
            return p[ptr[l]] if ptr[l] < len(p) else n + 1
        lane, jumps, i = 2, 0, 0
        while i < n:
            if obstacles[i + 1] != lane:
                i += 1
                continue
            # bị chặn ở i+1: nhảy ngang tại i sang làn hợp lệ có chướng ngại kế tiếp xa nhất
            cands = [l for l in (1, 2, 3) if l != lane and obstacles[i] != l]
            lane = max(cands, key=lambda l: next_obstacle(l, i))
            jumps += 1
        return jumps

for obs, want in examples:
    print(SolutionGreedy().minSideJumps(obs), want)

2 2
0 0
2 2


**Độ phức tạp:** `O(n)` (mỗi con trỏ chỉ tiến), bộ nhớ `O(số chướng ngại)`. Vì sao `cands` không rỗng? Tại điểm `i` có tối đa một chướng ngại; làn hiện tại `lane` trống tại `i` (ếch đang đứng), làn `obstacles[i+1] = lane`... hai làn còn lại có tối đa một làn bị chặn tại `i` ⇒ còn ít nhất một làn.

## 8. TRACE từng bước

Trace Cách 3 (DP) trên Ví dụ 1 `obstacles = [0,1,2,3,0]` (∞ = không đứng được):

| điểm | chướng ngại | sau bước 1 (chặn) | `min + 1` | sau bước 2 (nhảy ngang) |
|---|---|---|---|---|
| 0 | — | — | — | `[1, 0, 1]` |
| 1 | làn 1 | `[∞, 0, 1]` | 1 | `[∞, 0, 1]` |
| 2 | làn 2 | `[∞, ∞, 1]` | 2 | `[2, ∞, 1]` |
| 3 | làn 3 | `[2, ∞, ∞]` | 3 | `[2, 3, ∞]` |
| 4 | — | `[2, 3, ∞]` | 3 | `[2, 3, 3]` |

Đáp án `min = 2`. Ý nghĩa: ếch nhảy sang làn 3 (điểm 0 hoặc 1), đi tới điểm 2, nhảy sang làn 1 (tại điểm 2, làn 1 trống), đi tới cuối.

In [13]:
def trace_dp(obstacles):
    fmt = lambda a: "[" + ", ".join("∞" if x == inf else str(x) for x in a) + "]"
    dp = [1, 0, 1]
    print(f"điểm 0: {fmt(dp)}")
    for i in range(1, len(obstacles)):
        v = obstacles[i]
        if v: dp[v - 1] = inf
        after_block = dp[:]
        best = min(dp) + 1
        for l in range(3):
            if l != v - 1: dp[l] = min(dp[l], best)
        print(f"điểm {i}: chặn làn {v or '-'} -> {fmt(after_block)} | min+1 = {best} -> {fmt(dp)}")
    print("đáp án:", min(dp))
    return min(dp)

trace_dp([0, 1, 2, 3, 0])

điểm 0: [1, 0, 1]
điểm 1: chặn làn 1 -> [∞, 0, 1] | min+1 = 1 -> [∞, 0, 1]
điểm 2: chặn làn 2 -> [∞, ∞, 1] | min+1 = 2 -> [2, ∞, 1]
điểm 3: chặn làn 3 -> [2, ∞, ∞] | min+1 = 3 -> [2, 3, ∞]
điểm 4: chặn làn - -> [2, 3, ∞] | min+1 = 3 -> [2, 3, 3]
đáp án: 2


2

Trace Cách 4 (tham lam) trên Ví dụ 3 để thấy lựa chọn "xa nhất":

In [14]:
def trace_greedy(obstacles):
    n = len(obstacles) - 1
    def next_obs(l, i):
        for j in range(i + 1, n + 1):
            if obstacles[j] == l:
                return j
        return n + 1
    lane, jumps, i = 2, 0, 0
    while i < n:
        if obstacles[i + 1] != lane:
            i += 1; continue
        cands = {l: next_obs(l, i) for l in (1, 2, 3) if l != lane and obstacles[i] != l}
        new = max(cands, key=cands.get)
        jumps += 1
        print(f"tại điểm {i}, làn {lane} bị chặn ở {i+1}; ứng viên (làn: chướng ngại kế) {cands} -> nhảy sang {new}")
        lane = new
    print("số lần nhảy:", jumps)
    return jumps

trace_greedy([0, 2, 1, 0, 3, 0])

tại điểm 0, làn 2 bị chặn ở 1; ứng viên (làn: chướng ngại kế) {1: 2, 3: 4} -> nhảy sang 3
tại điểm 3, làn 3 bị chặn ở 4; ứng viên (làn: chướng ngại kế) {1: 6, 2: 6} -> nhảy sang 1
số lần nhảy: 2


2

## 9. Code nộp LeetCode

In [15]:
from math import inf
from typing import List

class Solution:
    def minSideJumps(self, obstacles: List[int]) -> int:
        dp = [1, 0, 1]                      # điểm 0: làn 1, 2, 3
        for v in obstacles[1:]:
            if v:
                dp[v - 1] = inf             # 1) không đứng được trên làn có chướng ngại
            best = min(dp) + 1
            for lane in range(3):           # 2) nhảy ngang tại điểm này
                if lane != v - 1 and dp[lane] > best:
                    dp[lane] = best
        return min(dp)

for obs, want in examples:
    assert Solution().minSideJumps(obs) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `D_i[l]` = số lần nhảy ít nhất để đứng ở `(i, l)` (∞ nếu không thể).

**Mệnh đề:** sau khi xử lý điểm `i`, `dp = D_i`.

*Cơ sở `i = 0`:* đứng ở làn 2 không tốn gì; làn 1, 3 cần ít nhất một lần nhảy và một lần là đủ ⇒ `[1, 0, 1]`.

*Bước `i − 1 → i`:* một cách tối ưu để đứng ở `(i, l)` có bước cuối là:
- **đi thẳng** từ `(i−1, l)` — hợp lệ ⇔ `obstacles[i] ≠ l`, chi phí `D_{i−1}[l]`;
- hoặc **nhảy ngang** tại điểm `i` từ `(i, l')` — và ta có thể giả sử trạng thái `(i, l')` đạt được bằng cách **đi thẳng** (nhảy hai lần liên tiếp tại một điểm luôn thay được bằng một lần nhảy thẳng sang làn đích). Chi phí `D_{i−1}[l'] + 1` với `l'` không bị chặn tại `i`.

Bước 1 của DP cho `dp[l] = D_{i−1}[l]` nếu `l` trống tại `i`, ∞ nếu bị chặn — đúng là chi phí "đi thẳng". Bước 2 lấy `min(dp) + 1` trên đúng các `l'` trống tại `i` (vì làn bị chặn đã là ∞) và chỉ cập nhật làn đích trống. ⇒ `dp = D_i`.

Đáp án `min(D_n)`. Luôn hữu hạn vì mỗi điểm có ≤ 1 chướng ngại nên luôn có ít nhất 2 làn trống.

**Tham lam (Cách 4) đúng** bằng lập luận trao đổi: giả sử lời giải tối ưu ở lần nhảy bắt buộc tại điểm `i` chọn làn `a`, còn tham lam chọn làn `g` có chướng ngại kế tiếp xa hơn hoặc bằng. Đoạn đi trên `a` tới khi lời giải tối ưu nhảy tiếp (trước chướng ngại của `a`) cũng đi được trên `g` (chướng ngại của `g` còn xa hơn); tại điểm lời giải tối ưu nhảy, tham lam hoặc chưa cần nhảy, hoặc nhảy cùng lúc — số lần nhảy không nhiều hơn. Quy nạp theo số lần nhảy.

## 11. Phân tích Time / Space

Gọi `n = len(obstacles) − 1 ≤ 5·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. DFS vét cạn | `O(3ⁿ)` | `O(n)` stack |
| 2. 0-1 BFS | `O(n)` | `O(n)` |
| 3. DP cuộn | `O(n)` | `O(1)` |
| 4. Tham lam | `O(n)` | `O(số chướng ngại)` |

In [16]:
rng = random.Random(1824)
n = 5 * 10**5
big = [0] + [rng.choice([0, 1, 2, 3]) for _ in range(n - 1)] + [0]
zigzag = [0] + [(i % 3) + 1 for i in range(n - 1)] + [0]       # chướng ngại luân phiên mọi điểm
for label, data in [("ngẫu nhiên", big), ("luân phiên", zigzag)]:
    res = []
    for name, f in [("0-1 BFS", side_jumps_01bfs), ("DP", SolutionDP().minSideJumps),
                    ("Tham lam", SolutionGreedy().minSideJumps), ("Nộp", Solution().minSideJumps)]:
        t = time.perf_counter(); r = f(data); res.append(r)
        print(f"[{label}] {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1

[ngẫu nhiên] 0-1 BFS: 111503 trong 537 ms
[ngẫu nhiên] DP: 111503 trong 108 ms


[ngẫu nhiên] Tham lam: 111503 trong 143 ms
[ngẫu nhiên] Nộp: 111503 trong 106 ms


[luân phiên] 0-1 BFS: 499998 trong 438 ms
[luân phiên] DP: 499998 trong 94 ms


[luân phiên] Tham lam: 499998 trong 330 ms
[luân phiên] Nộp: 499998 trong 94 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [17]:
sols = [Solution(), SolutionDP(), SolutionGreedy()]
edge = examples + [
    ([0, 0], 0),                     # n = 1
    ([0, 2, 0], 1),                  # chặn ngay trước mặt
    ([0, 1, 3, 0], 0),               # không bao giờ chặn làn 2
    ([0, 2, 2, 2, 0], 1),            # nhảy một lần là xong
    ([0, 2, 1, 2, 3, 2, 1, 0], 3),
    ([0, 0, 2, 0, 0, 0], 1),         # chỉ nhảy khi gần tới chướng ngại cũng được
    ([0, 2, 3, 1, 2, 3, 1, 0], 5),   # luân phiên dày
]
for obs, want in edge:
    assert side_jumps_brute(obs) == want, (obs, side_jumps_brute(obs))
    assert side_jumps_01bfs(obs) == want
    for s in sols:
        assert s.minSideJumps(obs) == want, (obs, s)
    print("OK", obs, "->", want)

OK [0, 1, 2, 3, 0] -> 2
OK [0, 1, 1, 3, 3, 0] -> 0
OK [0, 2, 1, 0, 3, 0] -> 2
OK [0, 0] -> 0
OK [0, 2, 0] -> 1
OK [0, 1, 3, 0] -> 0
OK [0, 2, 2, 2, 0] -> 1
OK [0, 2, 1, 2, 3, 2, 1, 0] -> 3
OK [0, 0, 2, 0, 0, 0] -> 1
OK [0, 2, 3, 1, 2, 3, 1, 0] -> 5


### 12.2 Ngẫu nhiên có seed: so với vét cạn và với 0-1 BFS

In [18]:
rng = random.Random(42)
for _ in range(2000):
    n = rng.randint(1, 12)
    obs = [0] + [rng.choice([0, 1, 2, 3, 2]) for _ in range(n - 1)] + [0]
    want = side_jumps_brute(obs)
    assert side_jumps_01bfs(obs) == want
    for s in sols:
        assert s.minSideJumps(obs) == want, obs
for _ in range(50):
    n = rng.randint(100, 3000)
    obs = [0] + [rng.choice([0, 1, 2, 3]) for _ in range(n - 1)] + [0]
    want = side_jumps_01bfs(obs)
    assert all(s.minSideJumps(obs) == want for s in sols)
print("2000 case nhỏ (vét cạn) + 50 case vừa (0-1 BFS): khớp")

2000 case nhỏ (vét cạn) + 50 case vừa (0-1 BFS): khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tính `min(dp)` **trước** khi chặn làn có chướng ngại

In [19]:
def wrong_order(obstacles):
    dp = [1, 0, 1]
    for v in obstacles[1:]:
        best = min(dp) + 1                         # SAI: có thể lấy làn sắp bị chặn
        if v:
            dp[v - 1] = inf
        for l in range(3):
            if l != v - 1:
                dp[l] = min(dp[l], best)
    return min(dp)

for obs, want in examples + [([0, 2, 0], 1)]:
    print(obs, "sai:", wrong_order(obs), "| đúng:", Solution().minSideJumps(obs))

[0, 1, 2, 3, 0] sai: 1 | đúng: 2
[0, 1, 1, 3, 3, 0] sai: 0 | đúng: 0
[0, 2, 1, 0, 3, 0] sai: 2 | đúng: 2
[0, 2, 0] sai: 1 | đúng: 1


**Vì sao sai:** `min(dp)` lúc đó còn tính làn `v` — làn mà ếch **không thể** đứng tại điểm này — như thể ếch có thể đi thẳng vào chướng ngại rồi mới nhảy. (Ví dụ 1 bị sai — ra 1 thay vì 2; Ví dụ 2, 3 và `[0,2,0]` **tình cờ** vẫn đúng, nên chỉ test bằng ví dụ đề là không đủ.) **Sửa:** chặn trước, `min` sau.

### Lỗi 2 — Nghĩ rằng chỉ nhảy được sang làn **kề**, nhảy 1 → 3 tốn 2

In [20]:
def wrong_adjacent(obstacles):
    n = len(obstacles) - 1
    dist = {(0, 2): 0}; dq = deque([(0, 0, 2)])
    while dq:
        d, i, l = dq.popleft()
        if d > dist[(i, l)]: continue
        if i == n: return d
        if obstacles[i + 1] != l and d < dist.get((i + 1, l), inf):
            dist[(i + 1, l)] = d; dq.appendleft((d, i + 1, l))
        for l2 in (l - 1, l + 1):                  # SAI: chỉ làn kề
            if 1 <= l2 <= 3 and obstacles[i] != l2 and d + 1 < dist.get((i, l2), inf):
                dist[(i, l2)] = d + 1; dq.append((d + 1, i, l2))
    return -1

for obs in [[0, 2, 1, 0], [0, 1, 2, 0, 3, 2, 0]]:
    print(obs, "sai:", wrong_adjacent(obs), "| đúng:", Solution().minSideJumps(obs))

[0, 2, 1, 0] sai: 1 | đúng: 1
[0, 1, 2, 0, 3, 2, 0] sai: 3 | đúng: 2


**Vì sao sai:** đề nói rõ "nhảy sang làn khác (**kể cả không kề nhau**)", một lần nhảy từ làn 1 sang làn 3 tốn **1**. Trường hợp `[0,1,2,0,3,2,0]`: muốn đổi giữa làn 1 và làn 3 tại một điểm mà làn 2 bị chặn, bản "chỉ kề" phải đi qua làn 2 — không được — nên buộc phải nhảy ở chỗ khác, tốn 3 thay vì 2. (Ví dụ tự dựng, đã đối chiếu bằng lời giải đúng và vét cạn ở mục 12.)

### Lỗi 3 — Khởi tạo `dp = [0, 0, 0]`

In [21]:
def wrong_init(obstacles):
    dp = [0, 0, 0]                                 # SAI: ếch chỉ bắt đầu ở làn 2
    for v in obstacles[1:]:
        if v: dp[v - 1] = inf
        best = min(dp) + 1
        for l in range(3):
            if l != v - 1: dp[l] = min(dp[l], best)
    return min(dp)

for obs, want in examples + [([0, 2, 0], 1)]:
    print(obs, "sai:", wrong_init(obs), "| đúng:", Solution().minSideJumps(obs))

[0, 1, 2, 3, 0] sai: 1 | đúng: 2
[0, 1, 1, 3, 3, 0] sai: 0 | đúng: 0
[0, 2, 1, 0, 3, 0] sai: 1 | đúng: 2
[0, 2, 0] sai: 0 | đúng: 1


**Vì sao sai:** coi như ếch đứng sẵn ở làn 1 và 3 miễn phí. Muốn ở làn 1 hoặc 3 tại điểm 0 phải nhảy một lần ⇒ `[1, 0, 1]`.

### Lỗi 4 — Tham lam "nhảy sang làn trống bất kỳ"

In [22]:
def wrong_greedy_any(obstacles):
    n = len(obstacles) - 1
    lane, jumps = 2, 0
    for i in range(n):
        if obstacles[i + 1] == lane:
            lane = next(l for l in (1, 2, 3) if l != lane and obstacles[i] != l)   # SAI: làn đầu tiên tìm thấy
            jumps += 1
    return jumps

obs = [0, 2, 1, 0, 3, 0]
print("sai:", wrong_greedy_any(obs), "| đúng:", Solution().minSideJumps(obs))

sai: 3 | đúng: 2


**Vì sao sai:** ở Ví dụ 3, nhảy sang làn 1 (tìm thấy đầu tiên) thì 1 điểm sau lại bị chặn; làn 3 "thoáng" tới điểm 4. Tham lam chỉ đúng khi chọn làn có **chướng ngại kế tiếp xa nhất**.

### Lỗi 5 — DP đệ quy có ghi nhớ trên `n = 5·10⁵`

In [23]:
def dp_recursive(obstacles):
    n = len(obstacles) - 1
    @lru_cache(maxsize=None)
    def f(i, l):                                  # ít nhất từ (i, l) tới đích
        if i == n: return 0
        best = inf
        if obstacles[i + 1] != l: best = f(i + 1, l)
        for l2 in (1, 2, 3):
            if l2 != l and obstacles[i] != l2 and obstacles[i + 1] != l2:
                best = min(best, 1 + f(i + 1, l2))
        return best
    return f(0, 2)

obs_long = [0] * 3001
try:
    print(dp_recursive(obs_long))
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "| DP vòng lặp:", Solution().minSideJumps(obs_long))
print("đệ quy đúng trên input ngắn:", all(dp_recursive(o) == w for o, w in examples))

RecursionError: maximum recursion depth exceeded | DP vòng lặp: 0
đệ quy đúng trên input ngắn: True


**Vì sao sai:** logic đúng (trên input ngắn ra đúng), nhưng độ sâu đệ quy bằng `n` — vượt giới hạn 1000 ngay với `n = 3000`, chưa nói `5·10⁵`. **Sửa:** DP vòng lặp xuôi (Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Di chuyển trên lưới hẹp (ít hàng), chỉ tiến về một hướng** ⇒ DP theo cột với **vài trạng thái** (ở đây 3 làn), bộ nhớ `O(1)`.
- **Hai loại bước, chi phí 0 và 1** ⇒ 0-1 BFS là lời giải "an toàn" khi chưa nghĩ ra DP.
- Trong DP có "chặn" và "chuyển ngang tại cùng cột": luôn **chặn trước, chuyển sau**.
- Tham lam "đi thẳng tới khi bắt buộc, khi rẽ thì chọn hướng thoáng xa nhất" — giống bài *Jump Game II* / *Minimum Number of Refueling Stops*.
- `n` tới `5·10⁵` ⇒ tránh đệ quy, tránh mảng 2 chiều lớn nếu không cần.

## 15. Biến thể của bài

1. **`k` làn** thay vì 3 (chướng ngại `0..k`, mỗi điểm tối đa một): DP `k` trạng thái, `O(n·k)` (Bài tập 5).
2. **Chi phí nhảy phụ thuộc khoảng cách làn** (`|l − l'|`) ⇒ trong DP thay `min(dp) + 1` bằng `min(dp[l'] + |l − l'|)` (với `k` nhỏ vẫn `O(n·k²)`).
3. **Nhiều chướng ngại mỗi điểm** ⇒ có thể không tới được đích; DP trả ∞ ⇒ `-1`.
4. **Trả về đường đi** (các điểm nhảy và làn đích) — truy vết DP hoặc dùng tham lam (Bài tập 3).
5. Bài tương tự: LeetCode 1871 *Jump Game VII*, 2304 *Minimum Path Cost in a Grid*, 1301.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS vét cạn | thử mọi đường | `O(3ⁿ)` | `O(n)` | kiểm chứng `n ≤ 15` |
| 2. 0-1 BFS | đồ thị `(điểm, làn)` | `O(n)` | `O(n)` | tổng quát, dễ thêm luật |
| 3. DP cuộn | 3 trạng thái, chặn rồi nhảy | `O(n)` | `O(1)` | **nộp bài** |
| 4. Tham lam | nhảy khi bắt buộc, chọn làn thoáng xa nhất | `O(n)` | `O(#chướng ngại)` | trực quan, cho ra cả đường đi |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[0,0]`, `[0,2,0]`, `[0,2,2,3,0]`?

**Bài 2 (tính tay):** Lập bảng DP cho Ví dụ 3 `[0,2,1,0,3,0]` theo mẫu mục 8.

**Bài 3 (điền TODO):** Viết `jump_plan(obstacles)` trả về list `(điểm, làn_cũ, làn_mới)` của một phương án tối ưu (dùng tham lam).

**Bài 4 (tự viết lại):** Viết lại DP nhưng lưu mảng đầy đủ `D[i][l]` (không cuộn) và từ đó **truy vết** số lần nhảy tại từng điểm.

**Bài 5 (mini challenge):** Tổng quát hóa cho `k` làn: `obstacles[i] ∈ {0..k}`, ếch bắt đầu ở làn `s` tại điểm 0. Viết `side_jumps_k(obstacles, k, s)`.

In [24]:
def jump_plan(obstacles):
    # TODO
    pass

def side_jumps_table(obstacles):
    # TODO: trả về bảng D (n+1) x 3
    pass

def side_jumps_k(obstacles, k, s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `n = 1` không có chướng ngại | `[0,2,0]`: phải nhảy 1 lần | `[0,2,2,3,0]`: nhảy sang làn 1 ở điểm 0, đi thẳng ⇒ 1 |
| 2 | Bắt đầu `[1,0,1]` | Điểm 1 chặn làn 2: `[1,∞,1]` → nhảy ngang không đổi | Điểm 4 chặn làn 3 … đáp án 2 |
| 3 | Dùng `SolutionGreedy` | Mỗi lần nhảy ghi `(i, lane, new_lane)` | Kiểm: phát lại phương án phải hợp lệ |
| 4 | `D[0] = [1, 0, 1]` | Mỗi hàng: copy hàng trước, chặn, rồi nhảy ngang | `D[n]` cho đáp án |
| 5 | `dp = [1]*k`, `dp[s−1] = 0` | Chặn làn `v`, `best = min(dp) + 1` | Đối chiếu với 0-1 BFS tổng quát `k` làn |

## 19. Đáp án tham khảo

In [25]:
r1 = [Solution().minSideJumps(o) for o in ([0, 0], [0, 2, 0], [0, 2, 2, 3, 0])]
print("Bài 1:", r1); assert r1 == [0, 1, 1]
print("Bài 2:"); assert trace_dp([0, 2, 1, 0, 3, 0]) == 2

Bài 1: [0, 1, 1]
Bài 2:
điểm 0: [1, 0, 1]
điểm 1: chặn làn 2 -> [1, ∞, 1] | min+1 = 2 -> [1, ∞, 1]
điểm 2: chặn làn 1 -> [∞, ∞, 1] | min+1 = 2 -> [∞, 2, 1]
điểm 3: chặn làn - -> [∞, 2, 1] | min+1 = 2 -> [2, 2, 1]
điểm 4: chặn làn 3 -> [2, 2, ∞] | min+1 = 3 -> [2, 2, ∞]
điểm 5: chặn làn - -> [2, 2, ∞] | min+1 = 3 -> [2, 2, 3]
đáp án: 2


In [26]:
def jump_plan(obstacles):
    n = len(obstacles) - 1
    def next_obs(l, i):
        for j in range(i + 1, n + 1):
            if obstacles[j] == l: return j
        return n + 1
    lane, i, plan = 2, 0, []
    while i < n:
        if obstacles[i + 1] != lane:
            i += 1; continue
        new = max((l for l in (1, 2, 3) if l != lane and obstacles[i] != l), key=lambda l: next_obs(l, i))
        plan.append((i, lane, new)); lane = new
    return plan

def replay(obstacles, plan):
    n = len(obstacles) - 1; lane = 2; jumps = {i: (a, b) for i, a, b in plan}
    for i in range(n + 1):
        if i in jumps:
            a, b = jumps[i]
            assert a == lane and obstacles[i] != b; lane = b
        if i < n:
            assert obstacles[i + 1] != lane
    return True

def side_jumps_table(obstacles):
    D = [[1, 0, 1]]
    for v in obstacles[1:]:
        row = D[-1][:]
        if v: row[v - 1] = inf
        best = min(row) + 1
        for l in range(3):
            if l != v - 1: row[l] = min(row[l], best)
        D.append(row)
    return D

rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 30)
    obs = [0] + [rng.choice([0, 1, 2, 3]) for _ in range(n - 1)] + [0]
    plan = jump_plan(obs)
    assert replay(obs, plan) and len(plan) == Solution().minSideJumps(obs)
    assert min(side_jumps_table(obs)[-1]) == Solution().minSideJumps(obs)
print("Bài 3, 4: OK", jump_plan([0, 2, 1, 0, 3, 0]))

Bài 3, 4: OK [(0, 2, 3), (3, 3, 1)]


In [27]:
def side_jumps_k(obstacles, k, s):
    dp = [1] * k
    dp[s - 1] = 0
    for v in obstacles[1:]:
        if v: dp[v - 1] = inf
        best = min(dp) + 1
        for l in range(k):
            if l != v - 1 and dp[l] > best: dp[l] = best
    return min(dp)

def side_jumps_k_bfs(obstacles, k, s):
    n = len(obstacles) - 1
    dist = {(0, s): 0}; dq = deque([(0, 0, s)])
    while dq:
        d, i, l = dq.popleft()
        if d > dist[(i, l)]: continue
        if i == n: return d
        if obstacles[i + 1] != l and d < dist.get((i + 1, l), inf):
            dist[(i + 1, l)] = d; dq.appendleft((d, i + 1, l))
        for l2 in range(1, k + 1):
            if l2 != l and obstacles[i] != l2 and d + 1 < dist.get((i, l2), inf):
                dist[(i, l2)] = d + 1; dq.append((d + 1, i, l2))
    return inf

rng = random.Random(8)
for _ in range(600):
    k = rng.randint(2, 5); n = rng.randint(1, 25); s = rng.randint(1, k)
    obs = [0] + [rng.randint(0, k) for _ in range(n - 1)] + [0]
    assert side_jumps_k(obs, k, s) == side_jumps_k_bfs(obs, k, s)
    if k == 3 and s == 2:
        assert side_jumps_k(obs, 3, 2) == Solution().minSideJumps(obs)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dp = [1, 0, 1]                      # ếch ở làn 2, điểm 0
for v in obstacles[1:]:
    if v: dp[v - 1] = inf           # 1) CHẶN trước
    best = min(dp) + 1
    for l in range(3):              # 2) NHẢY NGANG sau
        if l != v - 1: dp[l] = min(dp[l], best)
return min(dp)
```

**Checklist:**
- [ ] Khởi tạo `[1, 0, 1]` (không phải `[0, 0, 0]`)?
- [ ] Chặn làn có chướng ngại **trước** khi tính `min(dp)`?
- [ ] Nhảy ngang sang **bất kỳ** làn khác (không chỉ làn kề), chi phí 1?
- [ ] Không cập nhật làn đang bị chặn ở bước nhảy ngang?
- [ ] Không đệ quy sâu `n`; bộ nhớ `O(1)`.